# Phase 3 RAG Notebook

Goal: turn the six small knowledge-base docs into searchable chunks.

Final tool later:

```python
retrieve(query, k=4)
```

Run this notebook slowly from top to bottom. Do not jump to embeddings until the chunk table looks clean.

## What This Notebook Checks

- `sources.csv` loads correctly.
- Every source has `licence_ok == True`.
- Every `.md` file exists.
- One `.md` file can have multiple source rows.
- Chunks keep finding and source metadata.
- Later cells build an OpenAI + Chroma retrieval index.

In [13]:
from pathlib import Path
import json
import re

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

KB_DIR = PROJECT_ROOT / "data" / "kb"
RAW_DIR = KB_DIR / "raw"
PROCESSED_DIR = KB_DIR / "processed"
INDEX_DIR = KB_DIR / "chroma"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rag"

SOURCES_PATH = KB_DIR / "sources.csv"
CHUNKS_PATH = PROCESSED_DIR / "chunks.parquet"

for path in [PROCESSED_DIR, INDEX_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(SOURCES_PATH)

e:\Project\RadScribe
e:\Project\RadScribe\data\kb\sources.csv


## Step 1: Load Sources

`sources.csv` is the licence and citation table. This is the safety gate before chunking.

In [14]:
sources = pd.read_csv(SOURCES_PATH)
sources

,id,finding,title,url,source_name,licence,licence_url,raw_text_path,accessed_date,licence_ok,notes
0,pleural_effusion_nhlbi,Pleural Effusion,"Pleural Disorders - Pleurisy, Pleural Effusion...",https://www.nhlbi.nih.gov/health/pleural-disor...,NHLBI / NIH,Public-Domain,https://www.usa.gov/government-copyright,data/kb/raw/pleural_effusion.md,2026-08-22,True,Used for a short educational summary about ple...
1,pleural_effusion_wikipedia,Pleural Effusion,Pleural effusion,https://en.wikipedia.org/wiki/Pleural_effusion,Wikipedia,CC-BY-SA-4.0,https://creativecommons.org/licenses/by-sa/4.0/,data/kb/raw/pleural_effusion.md,2026-08-22,True,"Used for X-ray appearance details, including b..."
2,cardiomegaly_nhlbi,Cardiomegaly,Heart Failure - What Is Heart Failure?,https://www.nhlbi.nih.gov/health/heart-failure,NHLBI / NIH,Public-Domain,https://www.usa.gov/government-copyright,data/kb/raw/cardiomegaly.md,2026-08-22,True,"Used for clinical context about heart failure,..."
3,cardiomegaly_wikipedia,Cardiomegaly,Cardiomegaly,https://en.wikipedia.org/wiki/Cardiomegaly,Wikipedia,CC-BY-SA-4.0,https://creativecommons.org/licenses/by-sa/4.0/,data/kb/raw/cardiomegaly.md,2026-08-22,True,"Used for definition, common associated causes,..."
4,atelectasis_wikipedia,Atelectasis,Atelectasis,https://en.wikipedia.org/wiki/Atelectasis,Wikipedia,CC-BY-SA-4.0,https://creativecommons.org/licenses/by-sa/4.0/,data/kb/raw/atelectasis.md,2026-08-22,True,"Used for definition, partial or total lung col..."
5,consolidation_pneumonia_cdc,Consolidation / Pneumonia,About Pneumonia,https://www.cdc.gov/pneumonia/about/index.html,CDC,Public-Domain,https://www.usa.gov/government-copyright,data/kb/raw/consolidation_pneumonia.md,2026-08-22,True,"Used for pneumonia definition, infectious caus..."
6,consolidation_pneumonia_wikipedia,Consolidation / Pneumonia,Pulmonary consolidation,https://en.wikipedia.org/wiki/Pulmonary_consol...,Wikipedia,CC-BY-SA-4.0,https://creativecommons.org/licenses/by-sa/4.0/,data/kb/raw/consolidation_pneumonia.md,2026-08-22,True,Used for radiographic consolidation concept: n...
7,edema_wikipedia,Edema,Pulmonary edema,https://en.wikipedia.org/wiki/Pulmonary_edema,Wikipedia,CC-BY-SA-4.0,https://creativecommons.org/licenses/by-sa/4.0/,data/kb/raw/edema.md,2026-08-22,True,Used for pulmonary edema definition and chest ...
8,pneumothorax_nhlbi,Pneumothorax,"Pleural Disorders - Pleurisy, Pleural Effusion...",https://www.nhlbi.nih.gov/health/pleural-disor...,NHLBI / NIH,Public-Domain,https://www.usa.gov/government-copyright,data/kb/raw/pneumothorax.md,2026-08-22,True,Used for pneumothorax definition as air or gas...
9,pneumothorax_wikipedia,Pneumothorax,Pneumothorax,https://en.wikipedia.org/wiki/Pneumothorax,Wikipedia,CC-BY-SA-4.0,https://creativecommons.org/licenses/by-sa/4.0/,data/kb/raw/pneumothorax.md,2026-08-22,True,"Used for diagnostic/imaging context, including..."


## Step 2: Validate Licences And Files

This is the important assert. If a source is not approved, the notebook should stop.

In [15]:
required_columns = [
    "id",
    "finding",
    "title",
    "url",
    "source_name",
    "licence",
    "licence_url",
    "raw_text_path",
    "accessed_date",
    "licence_ok",
    "notes",
]

missing_columns = sorted(set(required_columns) - set(sources.columns))
assert not missing_columns, f"Missing source columns: {missing_columns}"

allowed_licences = {"Public-Domain", "CC-BY-4.0", "CC-BY-SA-4.0"}
bad_licences = sorted(set(sources["licence"]) - allowed_licences)
assert not bad_licences, f"Unapproved licence tokens: {bad_licences}"

licence_ok = sources["licence_ok"].astype(str).str.lower().eq("true")
assert licence_ok.all(), "Some sources have licence_ok != True"

missing_files = []
for raw_path in sources["raw_text_path"].unique():
    path = PROJECT_ROOT / raw_path
    if not path.exists():
        missing_files.append(raw_path)
assert not missing_files, f"Missing raw docs: {missing_files}"

print("source rows:", len(sources))
print("raw docs:", sources["raw_text_path"].nunique())
print("findings:", sources["finding"].nunique())
print("licence check passed")

source rows: 10
raw docs: 6
findings: 6
licence check passed


In [16]:
sources.groupby("finding").agg(
    source_rows=("id", "count"),
    raw_docs=("raw_text_path", "nunique"),
    licences=("licence", lambda x: ", ".join(sorted(set(x)))),
)

,source_rows,raw_docs,licences
finding,,,
Atelectasis,1,1,CC-BY-SA-4.0
Cardiomegaly,2,1,"CC-BY-SA-4.0, Public-Domain"
Consolidation / Pneumonia,2,1,"CC-BY-SA-4.0, Public-Domain"
Edema,1,1,CC-BY-SA-4.0
Pleural Effusion,2,1,"CC-BY-SA-4.0, Public-Domain"
Pneumothorax,2,1,"CC-BY-SA-4.0, Public-Domain"


## Step 3: Load Raw Markdown Docs

One `.md` file can be supported by more than one source row. We attach the full source list to each document.

In [17]:
def read_markdown(path: Path) -> str:
    text = path.read_text(encoding="utf-8")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


docs = []
for raw_path, group in sources.groupby("raw_text_path", sort=True):
    path = PROJECT_ROOT / raw_path
    text = read_markdown(path)
    finding_values = sorted(set(group["finding"]))
    assert len(finding_values) == 1, f"One file has multiple findings: {raw_path} -> {finding_values}"

    docs.append(
        {
            "raw_text_path": raw_path,
            "finding": finding_values[0],
            "text": text,
            "source_ids": group["id"].tolist(),
            "source_titles": group["title"].tolist(),
            "source_urls": group["url"].tolist(),
            "licences": group["licence"].tolist(),
            "source_names": group["source_name"].tolist(),
        }
    )

docs_df = pd.DataFrame(docs)
docs_df[["finding", "raw_text_path", "source_ids"]]

,finding,raw_text_path,source_ids
0,Atelectasis,data/kb/raw/atelectasis.md,[atelectasis_wikipedia]
1,Cardiomegaly,data/kb/raw/cardiomegaly.md,"[cardiomegaly_nhlbi, cardiomegaly_wikipedia]"
2,Consolidation / Pneumonia,data/kb/raw/consolidation_pneumonia.md,"[consolidation_pneumonia_cdc, consolidation_pn..."
3,Edema,data/kb/raw/edema.md,[edema_wikipedia]
4,Pleural Effusion,data/kb/raw/pleural_effusion.md,"[pleural_effusion_nhlbi, pleural_effusion_wiki..."
5,Pneumothorax,data/kb/raw/pneumothorax.md,"[pneumothorax_nhlbi, pneumothorax_wikipedia]"


## Step 4: Chunk Documents

Chunks should be small enough to retrieve precisely, but large enough to make sense alone.

Target: about 3-5 sentences per chunk.

In [18]:
def split_sentences(text: str) -> list[str]:
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        lines.append(line)

    joined = " ".join(lines)
    joined = re.sub(r"\s+", " ", joined).strip()
    if not joined:
        return []

    return re.split(r"(?<=[.!?])\s+", joined)


def make_chunks(text: str, sentences_per_chunk: int = 4, overlap: int = 1) -> list[str]:
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    step = max(1, sentences_per_chunk - overlap)
    for start in range(0, len(sentences), step):
        chunk_sentences = sentences[start : start + sentences_per_chunk]
        if not chunk_sentences:
            continue
        chunk = " ".join(chunk_sentences).strip()
        if len(chunk.split()) >= 20:
            chunks.append(chunk)
        if start + sentences_per_chunk >= len(sentences):
            break
    return chunks


chunk_rows = []
for doc in docs:
    doc_id = Path(doc["raw_text_path"]).stem
    doc_chunks = make_chunks(doc["text"])
    for chunk_idx, chunk_text in enumerate(doc_chunks):
        chunk_rows.append(
            {
                "chunk_id": f"{doc_id}_{chunk_idx:03d}",
                "finding": doc["finding"],
                "text": chunk_text,
                "raw_text_path": doc["raw_text_path"],
                "source_ids": json.dumps(doc["source_ids"]),
                "source_titles": json.dumps(doc["source_titles"]),
                "source_urls": json.dumps(doc["source_urls"]),
                "licences": json.dumps(doc["licences"]),
                "source_names": json.dumps(doc["source_names"]),
            }
        )

chunks = pd.DataFrame(chunk_rows)
print("chunks:", len(chunks))
chunks[["chunk_id", "finding", "text"]].head(10)

chunks: 21


,chunk_id,finding,text
0,atelectasis_000,Atelectasis,Atelectasis means part or all of a lung is not...
1,atelectasis_001,Atelectasis,"Atelectasis can happen for several reasons, in..."
2,atelectasis_002,Atelectasis,"Depending on the location, this may include cr..."
3,cardiomegaly_000,Cardiomegaly,Cardiomegaly means the heart appears enlarged....
4,cardiomegaly_001,Cardiomegaly,The finding matters because an enlarged cardia...
5,cardiomegaly_002,Cardiomegaly,A common measurement is the cardiothoracic rat...
6,cardiomegaly_003,Cardiomegaly,"Follow-up with clinical history, prior imaging..."
7,consolidation_pneumonia_000,Consolidation / Pneumonia,Pneumonia is an infection of the lungs. It can...
8,consolidation_pneumonia_001,Consolidation / Pneumonia,Consolidation is a radiographic pattern where ...
9,consolidation_pneumonia_002,Consolidation / Pneumonia,It may look denser or whiter than normal aerat...


In [19]:
chunks.groupby("finding").agg(
    chunks=("chunk_id", "count"),
    avg_words=("text", lambda x: round(sum(len(t.split()) for t in x) / len(x), 1)),
)

,chunks,avg_words
finding,,
Atelectasis,3,79.7
Cardiomegaly,4,65.5
Consolidation / Pneumonia,4,58.8
Edema,3,62.3
Pleural Effusion,4,66.5
Pneumothorax,3,73.7


## Step 5: Read A Few Chunks

Read these before moving on. If chunks look weird, fix chunking now.

In [20]:
for row in chunks.head(6).to_dict("records"):
    print("=" * 80)
    print(row["chunk_id"], "|", row["finding"])
    print(row["text"])
    print("sources:", ", ".join(json.loads(row["source_titles"])))

atelectasis_000 | Atelectasis
Atelectasis means part or all of a lung is not fully expanded because the air spaces have collapsed or lost volume. It can involve a small region, a lobe, or a larger part of the lung. It is different from consolidation because atelectasis is mainly a loss of air and volume, while consolidation means air spaces are filled with material such as fluid or inflammatory exudate. Atelectasis can happen for several reasons, including airway blockage, compression from nearby fluid or air, shallow breathing, or postoperative changes.
sources: Atelectasis
atelectasis_001 | Atelectasis
Atelectasis can happen for several reasons, including airway blockage, compression from nearby fluid or air, shallow breathing, or postoperative changes. Small areas may cause few symptoms, while larger areas can be associated with shortness of breath, faster breathing, low oxygen, or reduced movement on the affected side. On chest X-ray, atelectasis is often considered when there is i

## Step 6: Save Chunks

This is the first real output of Phase 3.

In [21]:
assert len(chunks) > 0, "No chunks created. Stop and fix before saving."
chunks.to_parquet(CHUNKS_PATH, index=False)
print(f"Saved chunks to: {CHUNKS_PATH}")

Saved chunks to: e:\Project\RadScribe\data\kb\processed\chunks.parquet


## Step 7: OpenAI Embedding Setup

Run this only after chunks look good.

You need `OPENAI_API_KEY` in your environment or a local `.env` file. Never commit `.env`.

In [22]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed; using environment variables only")

api_key_present = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY present:", api_key_present)

OPENAI_API_KEY present: True


## Step 8: Build Chroma Index With OpenAI Embeddings

This cell costs a tiny amount of API money. Only run it when Step 7 says the API key is present.

In [23]:
EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_COLLECTION = "radscribe_kb_openai"

def embed_openai(texts: list[str]) -> list[list[float]]:
    from openai import OpenAI

    client = OpenAI()
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


assert api_key_present, "Set OPENAI_API_KEY before running this cell."

import chromadb

client = chromadb.PersistentClient(path=str(INDEX_DIR))
try:
    client.delete_collection(CHROMA_COLLECTION)
except Exception:
    pass

collection = client.create_collection(name=CHROMA_COLLECTION, metadata={"hnsw:space": "cosine"})

texts = chunks["text"].tolist()
embeddings = embed_openai(texts)

metadatas = []
for row in chunks.to_dict("records"):
    metadatas.append(
        {
            "finding": row["finding"],
            "raw_text_path": row["raw_text_path"],
            "source_ids": row["source_ids"],
            "source_titles": row["source_titles"],
            "source_urls": row["source_urls"],
            "licences": row["licences"],
        }
    )

collection.add(
    ids=chunks["chunk_id"].tolist(),
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas,
)

print("stored chunks:", collection.count())
print("collection:", CHROMA_COLLECTION)

stored chunks: 21
collection: radscribe_kb_openai


## Step 9: First Retrieval Function

This is the notebook version of the future `src/rag/retrieve.py` function.

In [24]:
def retrieve(query: str, k: int = 4) -> list[dict]:
    query_embedding = embed_openai([query])[0]
    result = collection.query(query_embeddings=[query_embedding], n_results=k)

    rows = []
    for i in range(len(result["ids"][0])):
        distance = result["distances"][0][i]
        rows.append(
            {
                "chunk_id": result["ids"][0][i],
                "text": result["documents"][0][i],
                "source": result["metadatas"][0][i]["source_titles"],
                "finding": result["metadatas"][0][i]["finding"],
                "score": 1 - distance,
            }
        )
    return rows


pd.DataFrame(retrieve("fluid around the lung", k=4))[["score", "finding", "chunk_id", "text"]]

,score,finding,chunk_id,text
0,0.537067,Pleural Effusion,pleural_effusion_000,Pleural effusion means extra fluid has collect...
1,0.499207,Consolidation / Pneumonia,consolidation_pneumonia_002,It may look denser or whiter than normal aerat...
2,0.495075,Pleural Effusion,pleural_effusion_002,"On chest X-ray, pleural effusion is usually co..."
3,0.452757,Edema,edema_002,Patchier alveolar opacities may be seen in non...


## Stop Here

After this works, the next notebook work is:

1. make `data/kb/eval_queries.csv`
2. calculate recall@3 and MRR
3. show two example retrievals

Then move the clean code into `src/rag/`.